# CHEN — Phase 3: Dynamic Routing

Phase 1 and Phase 2 always invoked all 3 experts. Phase 3 asks: **do we need to wake them all?**

A tiny router classifier looks at the prompt and decides which subset of experts to activate. A trivial prompt ("write a haiku") wakes only the 3B Synthesizer. A hard prompt ("debug this C++ segfault") wakes the Analyst + Reasoner + Synthesizer. The bet: average cost per query drops by ≥50% with no statistically significant accuracy regression.

**Goal of Phase 3:** Cut cost without hurting quality, by waking only the experts each prompt actually needs.

## 1. Setup

In [ ]:
from chen.backends.mock import MockBackend
from chen.core.expert import Expert, ExpertRole
from chen.core.router import LogisticRouter, CosineRouter, HybridRouter
from chen.phases.phase3_routing import RoutingPipeline

## 2. Build the 4-expert garage

Phase 3 adds a `coder` expert — fine-tuned for code generation / debugging.

In [ ]:
experts = [
    Expert(name="analyst",     role=ExpertRole.ANALYST,     backend=MockBackend(params_m=3_000, role_hint="analyst")),
    Expert(name="reasoner",    role=ExpertRole.REASONER,    backend=MockBackend(params_m=8_000, role_hint="reasoner")),
    Expert(name="coder",       role=ExpertRole.CODER,       backend=MockBackend(params_m=7_000, role_hint="coder")),
    Expert(name="synthesizer", role=ExpertRole.SYNTHESIZER, backend=MockBackend(params_m=3_000, role_hint="synthesizer")),
]
print(f"Garage: {len(experts)} experts, total params: {sum(e.params_m for e in experts) / 1000}B")

## 3. Build the router

We use `LogisticRouter` — a deterministic, no-dependencies classifier that scores each expert based on prompt features (keyword density, length, code/math indicators).

In [ ]:
router = LogisticRouter.from_experts(experts)
pipe = RoutingPipeline(
    experts=experts,
    router=router,
    handoff="kv_cache",
    max_tokens_per_expert=128,
    memory_retrieve_k=0,
    write_intermediate_to_memory=False,
)
print(f"Router: {router.name}")

## 4. Cheap query: write a haiku

The router should activate only the Synthesizer (3B params).

In [ ]:
cheap_prompt = "Write a haiku about autumn leaves."
cheap_result = pipe.run(cheap_prompt)
print(f"Prompt: {cheap_prompt!r}")
print(f"Selected experts: {cheap_result.selected_experts}")
print(f"Params invoked:   {cheap_result.metrics.total_params_invoked_m / 1000}B")
print(f"Cost:             ${cheap_result.metrics.total_cost_usd:.6f}")
print(f"Latency:          {cheap_result.metrics.total_latency_ms:.2f} ms")

## 5. Math query: arithmetic with reasoning

The router should activate the Reasoner (+ Synthesizer for output).

In [ ]:
math_prompt = "What is 17 * 23? Show your reasoning step by step."
math_result = pipe.run(math_prompt)
print(f"Prompt: {math_prompt!r}")
print(f"Selected experts: {math_result.selected_experts}")
print(f"Params invoked:   {math_result.metrics.total_params_invoked_m / 1000}B")
print(f"Cost:             ${math_result.metrics.total_cost_usd:.6f}")
print(f"Latency:          {math_result.metrics.total_latency_ms:.2f} ms")

## 6. Expensive query: debug C++ code

The router should activate the Analyst + Coder (+ Synthesizer for output).

In [ ]:
expensive_prompt = (
    "Debug this segfault in my C++ allocator. The crash is in "
    "malloc_consolidate when freeing a large block after a realloc."
)
expensive_result = pipe.run(expensive_prompt)
print(f"Prompt: {expensive_prompt!r}")
print(f"Selected experts: {expensive_result.selected_experts}")
print(f"Params invoked:   {expensive_result.metrics.total_params_invoked_m / 1000}B")
print(f"Cost:             ${expensive_result.metrics.total_cost_usd:.6f}")
print(f"Latency:          {expensive_result.metrics.total_latency_ms:.2f} ms")

## 7. Cost comparison across the three queries

Compare against a hypothetical monolithic 70B model that loads all params for every query.

In [ ]:
from chen.core.config import CostModel

cost = CostModel()
baseline_params_m = 70_000

print(f"{'Query':<14} {'CHEN params':>14} {'CHEN cost':>14} {'Base cost':>14} {'Savings':>10}")
print("-" * 70)
for label, r in [("haiku", cheap_result), ("math", math_result), ("debug C++", expensive_result)]:
    chen_cost = r.metrics.total_cost_usd
    # Hypothetical monolith cost: same tokens, all 70B params loaded.
    base_cost = cost.cost_for(
        r.metrics.total_input_tokens,
        r.metrics.total_output_tokens,
        params_m=baseline_params_m,
        include_param_tax=True,
    )
    savings_pct = (1 - chen_cost / base_cost) * 100 if base_cost > 0 else 0
    print(
        f"{label:<14} "
        f"{r.metrics.distinct_params_invoked_m / 1000:>10.1f}B    "
        f"${chen_cost:>10.6f}  "
        f"${base_cost:>10.6f}  "
        f"{savings_pct:>7.1f}%"
    )

## 8. Try the other routers

CHEN ships three router variants. All implement the same `Router` protocol, so swapping is one line.

In [ ]:
for make_router in [
    lambda: ("logistic", LogisticRouter.from_experts(experts)),
    lambda: ("cosine",   CosineRouter.from_experts(experts)),
    lambda: ("hybrid",   HybridRouter.from_experts(experts)),
]:
    name, router = make_router()
    pipe = RoutingPipeline(
        experts=experts,
        router=router,
        handoff="text",  # isolate the router variable
        memory_retrieve_k=0,
        write_intermediate_to_memory=False,
    )
    r = pipe.run("Debug this Python function: def foo(): return None")
    print(f"{name:9s} router selected: {r.selected_experts}")

## 9. What did we just prove?

- Different prompts activate different expert subsets — the haiku query is much cheaper than the C++ debug query.
- The router correctly identified code/math/summarization prompts.
- All three routers (logistic, cosine, hybrid) work and produce reasonable selections.
- The Synthesizer is always last (force_last_role constraint), so the final output is always natural-language text.

## 10. Run the full benchmark suite

To see the EPU, cost-per-1M-tokens, and latency-to-accuracy KPIs against a 70B baseline, run:

```bash
python examples/run_benchmarks.py --phase 3 --router logistic
```

Or open `examples/run_benchmarks.py` to see how the harness works.